In [24]:
!pip install thefuzz -q
import pandas as pd
import numpy as np
import json

from geopy.distance import geodesic
from thefuzz import process
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Goal: A travel site recommendation system based on desired user features, with numerical values on how important each category of feature is important to them, or a scale on the spectrum for their preference
## Some feature include:
## Budget, Nature, Climate, Level of Tourism, etc

In [25]:
df_cities = pd.read_csv("/content/drive/MyDrive/Everywhen/cities.csv", encoding="utf-8")
df_unesco = pd.read_csv("/content/drive/MyDrive/Everywhen/unesco.csv", encoding="utf-8")
df_tourism = pd.read_csv("/content/drive/MyDrive/Everywhen/tourism.csv", encoding="utf-8")

df_qol = pd.read_csv("/content/drive/MyDrive/Everywhen/qol.txt", sep="\t", header = None, encoding="utf-8")
df_qol.columns = [
    "rank", "city", "quality_of_life_index", "purchasing_power_index",
    "safety_index", "health_care_index", "cost_of_living_index",
    "property_price_to_income_ratio", "traffic_commute_time_index",
    "pollution_index", "climate_index"
]
df_qol.to_csv("numbeo_quality_of_life.csv", index=False)

In [47]:

df_unesco_s = df_unesco[['Name EN', 'Short Description EN', 'States Names', 'Region', 'Region Code', 'Coordinates']]
df_unesco_s.head()

# 1. Ensure your short descriptions and names are treated as lowercase strings
df_unesco_s['Short Description EN'] = df_unesco_s['Short Description EN'].fillna('').astype(str).str.lower()
df_unesco_s['Name EN'] = df_unesco_s['Name EN'].fillna('').astype(str).str.lower()

def count_local_unesco_sites(row):
    city_name = str(row['city']).lower()
    country_name = str(row['country']).lower()

    country_mask = df_unesco_s['States Names'].str.lower().str.contains(country_name, na=False)
    country_sites = df_unesco_s[country_mask]

    # Count how many times the city name appears in the Name or Short Description
    city_match_mask = (
        country_sites['Name EN'].str.contains(city_name, regex=False) |
        country_sites['Short Description EN'].str.contains(city_name, regex=False)
    )

    return city_match_mask.sum()


/tmp/ipykernel_4699/1165591240.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unesco_s['Short Description EN'] = df_unesco_s['Short Description EN'].fillna('').astype(str).str.lower()
/tmp/ipykernel_4699/1165591240.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unesco_s['Name EN'] = df_unesco_s['Name EN'].fillna('').astype(str).str.lower()


In [27]:
df_tourism.head()
city_fixes = {
    "MalmÃ¶": "Malmo",
    "KrakÃ³w": "Krakow",
    "MalÃ©": "Male",
    "San SebastiÃ¡n": "San Sebastian",
    "FÃ¨s": "Fes",
    "QuÃ©bec City": "Quebec City",
    "BogotÃ¡": "Bogota",
    "BaÃ±os": "Banos",
    "TromsÃ¸": "Tromso",
    "AsunciÃ³n": "Asuncion",
    "SÃ£o TomÃ©": "Sao Tome",
    "LomÃ©": "Lome",
    "El ChaltÃ©n": "El Chalten",
    "GalÃ¡pagos Islands": "Galapagos Islands",
    "GjirokastÃ«r": "Gjirokaster",
    "NukuÊ»alofa": "Nukualofa",
    "CÃ³rdoba": "Cordoba",
    "YaoundÃ©": "Yaounde",
    "San JosÃ©": "San Jose",
    "ReykjavÃ\xadk": "Reykjavik",
    "San CristÃ³bal de las Casas": "San Cristobal de las Casas",
    "GorÃ©e Island": "Goree Island",
    "LÃ¼beck": "Lubeck",
    "GuimarÃ£es": "Guimaraes",
    "ValparaÃ\xadso": "Valparaiso",
    "MedellÃ\xadn": "Medellin",
    "San AndrÃ©s": "San Andres",
    "BÃºzios": "Buzios",
    "FlorianÃ³polis": "Florianopolis",
    "ViÃ±a del Mar": "Vina del Mar",
}

for broken, fixed in city_fixes.items():
    df_cities["city"] = df_cities["city"].str.replace(broken, fixed, regex=False)

df_cities = df_cities.drop_duplicates(subset=["city", "country"])


In [28]:
df_cities.sample()
df_cities = df_cities.drop(columns = ['id', 'budget_level'])
df_qol.drop(columns = ['health_care_index', 'property_price_to_income_ratio', 'purchasing_power_index'])
df_qol.head()
df_qol['country'] = df_qol['city'].str.split(',').str[-1].str.strip()
df_qol['city']    = df_qol['city'].str.split(',').str[0].str.strip()

,rank,city,quality_of_life_index,purchasing_power_index,safety_index,health_care_index,cost_of_living_index,property_price_to_income_ratio,traffic_commute_time_index,pollution_index,climate_index,country
0,1,The Hague (Den Haag),233.0,164.8,78.9,82.6,73.9,6.1,20.5,17.9,90.6,Netherlands
1,2,Eindhoven,217.9,151.4,77.4,74.8,70.3,7.2,23.5,20.7,85.4,Netherlands
2,3,Rotterdam,216.0,149.8,70.2,78.6,73.3,6.0,21.3,23.8,87.9,Netherlands
3,4,Stuttgart,215.1,187.3,66.0,76.6,69.4,6.5,25.2,36.8,81.1,Germany
4,5,Utrecht,213.0,155.9,73.2,78.6,76.1,7.4,24.7,28.8,87.2,Netherlands


In [29]:
def fuzzy_match(city, choices, threshold=80):
    match, score = process.extractOne(city, choices)
    return match if score >= threshold else None

qol_cities = df_qol["city"].tolist()

df_cities["city_matched"] = df_cities["city"].apply(
    lambda x: fuzzy_match(x, qol_cities)
)

df_merged = df_cities.merge(df_qol, left_on="city_matched", right_on="city", how="left")
matched   = df_merged["quality_of_life_index"].notna().sum()
df_merged = df_merged.drop(columns = ['country_y', 'city_y'])


In [30]:
tourism_lookup = df_tourism.set_index("country")["MostVisited_NumOfArrivals_WorldBank"].to_dict()
df_merged = df_merged.rename(columns={"city_x": "city", "country_x": "country"})
df_merged["popularity"] = df_merged["country"].map(tourism_lookup)

df_merged["popularity_score"] = np.log1p(df_merged["popularity"].fillna(0))
df_merged["popularity_score"] = (
    df_merged["popularity_score"] - df_merged["popularity_score"].min()
) / (df_merged["popularity_score"].max() - df_merged["popularity_score"].min())
threshold = df_merged["popularity_score"].quantile(0.35)
df_model = df_merged[df_merged["popularity_score"] >= threshold]


# Map the localized count directly onto main dataframe
df_merged['unesco_count'] = df_merged.apply(count_local_unesco_sites, axis=1)

print(df_merged[['city', 'country', 'unesco_count']].sort_values(by='unesco_count', ascending=False).head(10))


def parse_unesco_coords(coord_str):
    try:
        # Example if format is simple string: "48.8566, 2.3522"
        lat, lon = map(float, coord_str.split(','))
        return lat, lon
    except:
        return None

def count_nearby_unesco(city_row, unesco_df, radius_km=50):
    city_coords = (city_row['latitude'], city_row['longitude']) # Columns from your city database
    count = 0

    for _, unesco_row in unesco_df.iterrows():
        u_coords = parse_unesco_coords(unesco_row['Coordinates'])
        if u_coords:
            distance = geodesic(city_coords, u_coords).km
            if distance <= radius_km:
                count += 1
    return count

# Apply across dataset
df_merged['unesco_count'] = df_merged.apply(lambda row: count_nearby_unesco(row, df_unesco_s), axis=1)



               city   country  unesco_count
372         Beijing     China             7
122     Mexico City    Mexico             6
115          Sicily     Italy             5
461          Moscow    Russia             5
109            Rome     Italy             4
51   Rio de Janeiro    Brazil             3
25            Kyoto     Japan             3
281            Nara     Japan             3
175          Mumbai     India             3
338          Lisbon  Portugal             3


In [31]:
def weighted_cosine_similarity(user_vector, city_vector, weights=None):
    """
    Computes cosine similarity ignoring NaN features in the city vector.
    Weights allow some features to matter more than others.
    """
    user  = np.array(user_vector, dtype=float)
    city  = np.array(city_vector, dtype=float)

    # Find valid (non-NaN) indices
    valid = ~np.isnan(city) & ~np.isnan(user)

    if valid.sum() == 0:
        return 0.0  # no overlap at all

    u = user[valid]
    c = city[valid]
    w = weights[valid] if weights is not None else np.ones(len(u))

    # Weighted cosine similarity on valid features only
    dot       = np.dot(u * w, c)
    magnitude = np.linalg.norm(u * w) * np.linalg.norm(c)

    return dot / magnitude if magnitude != 0 else 0.0


def recommend(user_prefs, df, feature_cols, weights=None, top_n=10):
    user_vector = np.array([user_prefs.get(f, np.nan) for f in feature_cols])
    weights_arr = np.array(weights) if weights else None

    scores = df[feature_cols].apply(
        lambda row: weighted_cosine_similarity(user_vector, row.values, weights_arr),
        axis=1
    )

    df = df.copy()
    df["match_score"] = (scores * 100).round(1)
    return df.sort_values("match_score", ascending=False).head(top_n)

In [48]:
feature_cols = [
    "unesco_count","nature", "beaches", "urban", "seclusion", "safety_index", "cost_of_living_index",
    "climate_index", "popularity_score", 'pollution_index'
]

for col in feature_cols:
    non_null = df_merged[col].dropna()
    if len(non_null) > 0:
        min_val, max_val = non_null.min(), non_null.max()
        if max_val > min_val:
            df_merged[col] = (df_merged[col] - min_val) / (max_val - min_val) * 100

#higher cost of living index = more expensive
#climate index, higher = hotter
# Optional: weight some features more than others (must match feature_cols order)
weights = [1, 1,1, 3, 2, 2, 1.5, 1, 3, 1]

user_prefs = {
    "unesco_count":          100,
    "nature":                10,
    "beaches":               20,
    "urban":                100,
    "seclusion":             20,
    "safety_index":          75,
    "cost_of_living_index":  80,
    "climate_index":         40,
    "popularity_score":      100,
    "pollution_index":       75,
}

regions = ["europe"]

df_filtered = df_merged[df_merged["region"].str.lower().isin(regions)]

results = recommend(user_prefs, df_filtered, feature_cols, weights, top_n=15)


print(results[["city", "country", "match_score"]])

           city         country  match_score
177     Seville           Spain         98.2
518     Leipzig         Germany         95.5
379      Venice           Italy         95.2
516     Granada           Spain         94.6
345      Bilbao           Spain         94.6
422     Cardiff  United Kingdom         94.5
207      London  United Kingdom         93.6
517        Linz         Austria         93.3
229    Bordeaux          France         92.4
230  Strasbourg          France         92.3
89       Verona           Italy         92.3
86       Berlin         Germany         92.1
165      Madrid           Spain         91.4
371      Prague  Czech Republic         90.8
505        Pisa           Italy         90.7


In [49]:
ui_cols = [
    "city", "country", "region",
    "unesco_count", "nature", "beaches", "urban", "seclusion",
    "safety_index", "cost_of_living_index", "climate_index",
    "popularity_score", "pollution_index", "short_description"
]

df_export = df_merged[ui_cols].copy()
df_export["region"] = df_export["region"].str.lower().str.replace(" ", "_")

#  Fill missing descriptions with empty strings so JavaScript doesn't break
df_export["short_description"] = df_export["short_description"].fillna("")
df_export = df_export.replace({float('nan'): None})


df_export["short_description"] = (
    df_export["short_description"]
    .astype(str)
    .str.replace("cafÃ©", "café", case=False, regex=False)
    .str.replace("Ã©", "é", regex=False)
    .str.replace("Ã¡", "á", regex=False)
    .str.replace("Ã³", "ó", regex=False)
    .str.replace("Ãº", "ú", regex=False)
    .str.replace("Ã±", "ñ", regex=False)
    .str.replace("Ã", "í", regex=False)
)

# generate and save the JSON
json_data = json.dumps(df_export.to_dict(orient="records"), ensure_ascii=False)
with open("city_data.json", "w", encoding="utf-8") as f:
    f.write(json_data)

print("Successfully generated 'city_data.json' with short descriptions!")

Successfully generated 'city_data.json' with short descriptions!
